In [1]:
import pandas as pd
import networkx as nx 
from matplotlib import pyplot as plt
import numpy as np

In [2]:
edge_list = pd.read_csv(r"D:\commo\code\4_kumu_struct\edge_list.csv")
node1 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list1.csv")
node2 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list2.csv")
node3 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list3.csv")
others = pd.read_csv(r"D:\commo\code\4_kumu_struct\others.csv")
node1.head()

,Label,Type,Description,tier,commodity_type,commodity_focus,acra_uen,hq_country,est_revenue,global_headcount,est_trade_volume,ownership_struct,exchange,sg_address
0,trafigura group,"trader, lender",Latest acquisition (after Sep 2025): Greenergy...,3.0,multi (>2 types),oil | petroleum | natural gas | LNG | metals |...,TRAFIGURA GROUP PTE. LTD. / 201017488D (record...,Singapore,Trafigura Group: 240.3 USD bn FY2025 (trafigur...,5011 (linkedin),approx. 462 million metric tons (2025 annual r...,private,NaN,"10 Collyer Quay, #29-01/05, Ocean Financial Ce..."
1,vitol asia,trader,"Its largest operations are in Geneva, Houston,...",3.0,multi (>2 types),crude oil | petroleum | LNG | natural gas | bi...,VITOL ASIA PTE LTD. / 199001917Z (recordowl),Switzerland,Turnover of $343 billion FY2025 (company website),1990 (linkedin),total energy trade volume: 605 million tonnes ...,private,NaN,128 BEACH ROAD #28-01 GUOCO MIDTOWN OFFICE SIN...
2,mercuria holdings,others,Mercuria Holdings (Singapore) Pte. Ltd. is a g...,3.0,multi (>2 types),energy | metals | soft/agricultural,MERCURIA HOLDINGS (SINGAPORE) PTE. LTD. / 2018...,Switzerland,$618.7 Million (zoom.info),1380 (linkedin),6M+ barrels of oil equivalent (BOE) per day (c...,private,NaN,"12 Marina View, #26-01, Asia Square Tower 2, S..."
3,gunvor singapore,trader,one of the world's largest independent commodi...,3.0,energy,crude oil | refined petroleum products | natur...,GUNVOR SINGAPORE PTE. LTD. / 200606959K (acra ...,Switzerland,US $144 billion (company website),1035 (linkedin),253 million MT (company website),private,NaN,"128 Beach Road, #29-01, Guoco Midtown Office, ..."
4,louis dreyfus company asia,trader,"LDC is purely agricultural, no energy commodit...",3.0,soft/agricultural,grains | oilseeds | coffee | cotton | sugar | ...,LOUIS DREYFUS COMPANY ASIA PTE. LTD. / 1993065...,Netherlands,Net Sales: US$ 53.2bn FY2025 (2025.12 LDC - In...,"13,359 (linkedin)",~100 million tonnes of products shipped annual...,private,NaN,12 MARINA BOULEVARD #33-03 MARINA BAY FINANCIA...


In [3]:
node1.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Tier':'tier',
    'Commodity Type':'commodity_type',
    'Commodity Focus':'commodity_focus',
    'Legal Name / ACRA UEN':'acra_uen',
    'HQ Country':'hq_country',
    'Estimated Revenue':'est_revenue',
    'Headcount (Global)':'global_headcount',
    'Trade Volume (Est.)':'est_trade_volume',
    'Ownership Structure':"ownership_struct",
    'Exchange':'exchange',
    'APAC offices':'sg_address'
}, inplace=True)
node2.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Scope':'scope',
    'Office Location(s) in APAC':'sg_address',
    'Practice Areas':'practice_areas',
    'Known Client Base':'known_clients'
}, inplace=True)
node3.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Financier Type':'financieir_type',
    'Commodity Type':'commodity_type',
    'Geographic Reach':'geographic_reach',
    'Known Clients':'known_clients'
}, inplace=True)
others.rename(columns={
    'Label':'label',
    'Type':'role',
}, inplace=True)

In [4]:
# create undirected graph
UG = nx.Graph()
# add all nodes
# add all nodes
records1 = node1.set_index("label").to_dict("index")
records2 = node2.set_index("label").to_dict("index")
records3 = node3.set_index("label").to_dict("index")
records4 = others.set_index("label").to_dict("index")
UG.add_nodes_from(records1.items())
UG.add_nodes_from(records2.items())
UG.add_nodes_from(records3.items())
UG.add_nodes_from(others.items())

# add edges
edges = [(row.From, row.To, {'type':row.Type}) for row in edge_list[['From','To','Type']].itertuples()]
UG.add_edges_from(edges)
# UG['trafigura group']['trafigura carbon trading'] > {'type':'owns'}

def remove_isolated_nodes_undirected(G):
    G = G.copy()
    isolated = [n for n in G.nodes() if G.degree(n) == 0]
    G.remove_nodes_from(isolated)
    return G

UG = remove_isolated_nodes_undirected(UG)
print(UG.number_of_nodes(), "nodes remaining")

# community detection, run five times with different random seeds
run1 = nx.community.louvain_communities(UG, seed=123) # visualize
run2 = nx.community.louvain_communities(UG, seed=234)
run3 = nx.community.louvain_communities(UG, seed=345)
run4 = nx.community.louvain_communities(UG, seed=456)
run5 = nx.community.louvain_communities(UG, seed=567)

runs = {
    123: run1,
    234: run2,
    345: run3,
    456: run4,
    567: run5,
}

mods = []
no_communities = []
for seed, run in runs.items():
    mod = nx.community.modularity(UG, run)
    mods.append(mod)

five_runs_df = pd.DataFrame(columns=['seed','no_communities', 'modularity'])
five_runs_df['seed'] = list(runs.keys())
five_runs_df['no_communities'] = [len(run) for seed,run in runs.items()]
five_runs_df["modularity"] = [round(m, 4) for m in mods]
print(five_runs_df)

max_mod = five_runs_df["modularity"].max()
best_row = five_runs_df.loc[five_runs_df["modularity"] == max_mod].iloc[0]
print(f"Best run with highest modularity score: seed {int(best_row['seed'])}, modularity score {best_row['modularity']}")

# check run with seed 234 for the communities
# create a node list with label, type, community
rows = []
for idx, community in enumerate(runs[234]):
    names = []
    for name in community:
        rows.append({'Label':name, 'Community': idx})
original_comm = pd.DataFrame(rows)

label_lookups = pd.concat([
    node1.set_index('label')['role'], node2.set_index('label')['role'], node3.set_index('label')['role'], others.set_index('label')['role']])
original_comm.insert(1, "Type", original_comm["Label"].map(label_lookups))

# save original community and five_runs_df
original_comm

163 nodes remaining
   seed  no_communities  modularity
0   123              25      0.7625
1   234              26      0.7626
2   345              25      0.7625
3   456              25      0.7610
4   567              25      0.7625
Best run with highest modularity score: seed 234, modularity score 0.7626


,Label,Type,Community
0,trafigura beheer bv,others,0
1,trafigura group,"trader, lender",0
2,icbc singapore branch,lender,0
3,china construction bank singapore branch,lender,0
4,tarago operations,others,0
...,...,...,...
158,maybank singapore,lender,24
159,petrochina international singapore,trader,24
160,petrochina international,trader,24
161,haridass ho & partners,lawyer,25


In [5]:
size_per_community = pd.DataFrame(original_comm.groupby("Community")['Label'].agg('count'))
size_per_community.rename(columns={'Label':'Label Count'}, inplace=True)
size_per_community_sorted = size_per_community.sort_values(by="Label Count", ascending=False)
size_per_community_sorted

,Label Count
Community,
6,23
0,11
1,11
3,11
17,11
7,10
19,10
5,10
20,9


In [ ]:
size_per_community_sorted['Label Count'].median()

In [6]:
original_comm.to_csv(r"D:\commo\code\network_analysis\original_community_detection.csv", index=False)
five_runs_df.to_csv(r"D:\commo\code\network_analysis\five_runs_df.csv", index=False)

In [7]:
# create a directed graph
DG = nx.DiGraph()

# add all nodes
records1 = node1.set_index('label').to_dict('index')
records2 = node2.set_index("label").to_dict("index")
records3 = node3.set_index("label").to_dict("index")
DG.add_nodes_from(records1.items())
DG.add_nodes_from(records2.items())
DG.add_nodes_from(records3.items())

# add edges
edges = [(row.From, row.To, {'type':row.Type}) for row in edge_list[['From','To','Type']].itertuples()]
DG.add_edges_from(edges) # full edge list

In [8]:
# in-degree, out-degree
print()
in_degree_centrality = nx.in_degree_centrality(DG) # visualize
in_degree_centrality_sorted = sorted(list(in_degree_centrality.items()), key=lambda x: x[1], reverse=True)
print('Nodes with highest in-degree centrality:')
for label, score in in_degree_centrality_sorted[:10]:
    print(label, score)

print()
out_degree_centrality = nx.out_degree_centrality(DG) # visualize
out_degree_centrality_sorted = sorted(list(out_degree_centrality.items()), key=lambda x: x[1], reverse=True)
print('Nodes with highest out-degree centrality:')
for label, score in out_degree_centrality_sorted[:10]:
    print(label, score)

# betweenness centrality
print()
betweenness_centrality = nx.betweenness_centrality(DG) # visualize
betweenness_centrality_sorted = sorted(list(betweenness_centrality.items()), key=lambda x: x[1], reverse=True)
print('Nodes with highest betweenness centrality:')
for label, score in betweenness_centrality_sorted[:10]:
    print(label, score)

# pagerank v2 using networkx built-in function
DG_copy = DG.copy()
print()
print("Removing isolated nodes..")
all_nodes = list(DG_copy.nodes())
# DG.edges({node}) shows all the edges, len(DG.edges({node})) shows the no of edges that node has
for node in all_nodes:
    if DG_copy.in_degree(node) == 0 and DG_copy.out_degree(node) == 0:
        DG_copy.remove_node(node)
print('Remaining nodes and edges after removing isolated nodes:', DG_copy.number_of_nodes(), DG_copy.number_of_edges())  # 163 nodes remaining
# labelling nodes as integers
print("Relabelling nodes to integers...")
print()
# relabel nodes as integers
n_unique_nodes = len(set(DG_copy.nodes()))  # 296 nodes total
# create dict{'label':int}
node2int = dict(zip(set(DG_copy.nodes()), range(n_unique_nodes)))  # {'cofco international': 0,'hyundai corporation singapore': 1, etc}
# create an opp dict
int2node = {v: k for k, v in node2int.items()}  # {0: 'cofco international',1: 'hyundai corporation singapore', etc}
# relabel the directed graph with the node2int dict
DG_copy = nx.relabel_nodes(DG_copy, node2int)
pagerank_score = nx.pagerank(DG_copy, alpha=0.85)
pagerank_score_ranked = sorted(pagerank_score.items(), key=lambda x: x[1], reverse=True)
# for i in pagerank_score_ranked:
#    print(i[0], int2node[i[0]], i[1]) # idx, name, score
print('Nodes with highest pagerank score:')
for idx, score in pagerank_score_ranked[:10]:
    print(int2node[idx], score)

# constraints to get structural holes
print()
constraints = nx.constraint(DG_copy)
constraints_ranked = sorted(constraints.items(), key=lambda x: x[1], reverse=False)
print('Nodes with lowest constraint score:')
for idx, score in constraints_ranked[:10]:
    print(int2node[idx], score)


Nodes with highest in-degree centrality:
trafigura group 0.04421768707482993
gunvor group 0.02040816326530612
vitol group 0.017006802721088433
louis dreyfus company asia 0.013605442176870748
cofco international 0.013605442176870748
wilmar international 0.01020408163265306
unipec singapore 0.01020408163265306
glencore 0.01020408163265306
oversea-chinese banking corporation ocbc 0.01020408163265306
united overseas bank uob 0.01020408163265306

Nodes with highest out-degree centrality:
smbc 0.03401360544217687
tsmp law corporation 0.023809523809523808
trafigura group 0.02040816326530612
allen & gledhill 0.02040816326530612
lvm law chambers llc 0.02040816326530612
mufg 0.02040816326530612
hsbc 0.02040816326530612
incisive law llc 0.017006802721088433
standard chartered 0.017006802721088433
dbs bank 0.017006802721088433

Nodes with highest betweenness centrality:
trafigura group 0.00111443894964129
dbs bank 0.00029021847646908594
united overseas bank uob 0.00026700099835155907
gunvor group

In [9]:
# create df of original metric scores
all_labels = set(in_degree_centrality) | set(out_degree_centrality) | set(betweenness_centrality)
rows = []
for label in all_labels:
    rows.append({
        'Label': label,
        'In-degree': round(in_degree_centrality.get(label),4),
        'Out-degree': round(out_degree_centrality.get(label),4),
        'Betweenness': round(betweenness_centrality.get(label),4),
    })
original_metric_scores = pd.DataFrame(rows)
pagerank_df = pd.DataFrame([
    {'Label': int2node[idx], 'PageRank': round(score,4)} for idx, score in pagerank_score.items()
])
constraint_df = pd.DataFrame([
    {'Label': int2node[idx], 'Constraint': round(score,4)} for idx, score in constraints.items()
])

original_metric_scores = original_metric_scores.merge(pagerank_df, on='Label', how='left')
original_metric_scores = original_metric_scores.merge(constraint_df, on='Label', how='left')
original_metric_scores.insert(1, "Type", original_comm["Label"].map(label_lookups))

# save the original metric scores
original_metric_scores

,Label,Type,In-degree,Out-degree,Betweenness,PageRank,Constraint
0,focus law asia llc,others,0.0000,0.0034,0.0,0.0043,1.0
1,ince & co singapore,"trader, lender",0.0000,0.0068,0.0,0.0043,0.5
2,ossiano,lender,0.0000,0.0000,0.0,NaN,NaN
3,mri group,lender,0.0000,0.0000,0.0,NaN,NaN
4,aditya birla global trading,others,0.0000,0.0000,0.0,NaN,NaN
...,...,...,...,...,...,...,...
290,incomlend,NaN,0.0000,0.0000,0.0,NaN,NaN
291,glory bulk carriers,NaN,0.0034,0.0000,0.0,0.0061,1.0
292,cemcoa energy,NaN,0.0000,0.0000,0.0,NaN,NaN
293,bts tankers,NaN,0.0034,0.0000,0.0,0.0049,1.0


In [10]:
original_metric_scores.to_csv(r"D:\commo\code\network_analysis\original_metric_scores.csv", index=False)

# name of firms with top in-degree, out-degree, betweenness, pagerank
# Nodes with highest in-degree centrality: trafigura group, gunvor group, vitol group, louis dreyfus company asia, cofco international, 
#                                          wilmar international, unipec singapore, glencore, oversea-chinese banking corporation ocbc 
#                                          united overseas bank uob 
# Nodes with highest out-degree centrality: smbc, tsmp law corporation, trafigura group, allen & gledhill, lvm law chambers llc,
#                                           mufg, hsbc, incisive law llc, standard chartered, dbs bank 
# Nodes with highest betweenness centrality: trafigura group, dbs bank, united overseas bank uob, gunvor group, vitol group, 
#                                          sinar mas group, mercuria energy trading, wilmar international, trafigura beheer bv 
#                                          glencore
# Nodes with highest pagerank score: trafigura group, louis dreyfus company asia, standard chartered bank singapore scb, 
#                                    ed&f man capital markets singapore, shell eastern trading, mercuria energy trading,
#                                    wilmar sugar, cofco international singapore, petrochina international singapore 
#                                    unipec singapore 
# Nodes with lowest constraint score: trafigura group, smbc, gunvor group, united overseas bank uob, vitol group,
#                                     tsmp law corporation, dbs bank, allen & gledhill, lvm law chambers llc, mufg 


In [11]:
# collective metric of the directed graph

# number of nodes and edges
# node count
total_num_nodes = nx.number_of_nodes(DG)
# edge count
total_num_edges = nx.number_of_edges(DG)
print('Original number of nodes and edges: ',total_num_nodes, total_num_edges)
# ave in degree
ave_in_degree = sum(score for label,score in in_degree_centrality.items()) / total_num_nodes
# ave out degree
ave_out_degree = sum(score for label, score in out_degree_centrality.items()) / total_num_nodes
# density of directed graph
print()
density_dg = nx.density(DG) # visualize
# checks if a directed graph is weakly connected nx.is_weakly_connected
is_wcc = nx.is_weakly_connected(DG) # false 
# no of wcc nx.number_weakly_connected_components
num_wcc = nx.number_weakly_connected_components(DG)

# save the collective metric table
collective_metric = pd.DataFrame({'Collective Metrics': ['total_num_nodes','total_num_edges','ave_in_degree','ave_out_degree','density','is_weakly_connected','num_wcc'],
                                  'Value':[total_num_nodes,total_num_edges, round(ave_in_degree,4),round(ave_out_degree,4),round(density_dg,4),is_wcc,num_wcc]})
collective_metric.to_csv(r"D:\commo\code\network_analysis\collective_metric.csv",index=False)
collective_metric

Original number of nodes and edges:  295 170



,Collective Metrics,Value
0,total_num_nodes,295
1,total_num_edges,170
2,ave_in_degree,0.002
3,ave_out_degree,0.002
4,density,0.002
5,is_weakly_connected,False
6,num_wcc,150


In [12]:
# targeted attack robustness analysis
# deciding k value
DG_copy = DG.copy()
print("Removing isolated nodes..")
all_nodes = list(DG_copy.nodes())
# DG.edges({node}) shows all the edges, len(DG.edges({node})) shows the no of edges that node has
for node in all_nodes:
    if DG_copy.in_degree(node) == 0 and DG_copy.out_degree(node) == 0:
        DG_copy.remove_node(node)
print('Remaining nodes after removing isolated nodes:', DG_copy.number_of_nodes())  # 163 nodes remaining
print('Deciding K value with consideration of graph sparsity and reflecting real-world scenario:')
print(f"K = {round(0.02 * DG_copy.number_of_nodes(),0)} if remove 2% of remaining unisolated nodes: ")
print(f"K = {round(0.05 * DG_copy.number_of_nodes(),0)} if remove 5% of remaining unisolated nodes: ")

Removing isolated nodes..
Remaining nodes after removing isolated nodes: 163
Deciding K value with consideration of graph sparsity and reflecting real-world scenario:
K = 3.0 if remove 2% of remaining unisolated nodes: 
K = 8.0 if remove 5% of remaining unisolated nodes: 


In [13]:
def remove_topk(DG, name, scores, topk):
    reverse = 'constraints' not in name  # low constraint = "top" broker, so ascending sort for constraints
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=reverse)[:topk]
    topk_names = [n for n, score in ranked]
    return DG.subgraph(nodes=[n for n in DG.nodes if n not in topk_names])

k = [3,8]
scores = {'in_degree':in_degree_centrality, 'out_degree':out_degree_centrality, 'betweenness': betweenness_centrality, 'pagerank_score':pagerank_score, 'constraints':constraints}
new_graphs = {}
for n in k:
    for name, s in scores.items():
        if 'constraints' in name:
            key = f"DG_bottom{n}_{name}_removed"
        else:
            key = f"DG_top{n}_{name}_removed" 
        G_removed = remove_topk(DG, name, s, n)
        new_graphs[key] = G_removed
        # print(f'Number of nodes after top {n} nodes with high {name} removed: ',  G_removed.number_of_nodes())
print(new_graphs.keys())

dict_keys(['DG_top3_in_degree_removed', 'DG_top3_out_degree_removed', 'DG_top3_betweenness_removed', 'DG_top3_pagerank_score_removed', 'DG_bottom3_constraints_removed', 'DG_top8_in_degree_removed', 'DG_top8_out_degree_removed', 'DG_top8_betweenness_removed', 'DG_top8_pagerank_score_removed', 'DG_bottom8_constraints_removed'])


In [14]:
# for each new graph, recompute network analysis metrics, pagerank and constraint
new_graph_scores = {}
for g in new_graphs.keys():

    new_in_degree = nx.in_degree_centrality(new_graphs[g])
    new_out_degree = nx.out_degree_centrality(new_graphs[g])
    new_betweenness = nx.betweenness_centrality(new_graphs[g])

    # remove isolated nodes for page rank and contraint score
    new_graph_copy = new_graphs[g].copy()
    # print("Removing isolated nodes..")
    all_nodes = list(new_graph_copy.nodes())
    for node in all_nodes:
        if new_graph_copy.in_degree(node) == 0 and new_graph_copy.out_degree(node) == 0:
            new_graph_copy.remove_node(node)
    # print('Remaining nodes and edges after removing isolated nodes:', new_graph_copy.number_of_nodes(), new_graph_copy.number_of_edges())  # 163 nodes remaining
    pagerank_score = nx.pagerank(new_graph_copy, alpha=0.85)
    ranked = sorted(pagerank_score.items(), key=lambda x: x[1], reverse=True)

    constraints = nx.constraint(new_graph_copy)
    constraints_ranked = sorted(constraints.items(), key=lambda x: x[1], reverse=False)

    score_details = {}
    score_details['new_in_degree'] = new_in_degree
    score_details['new_out_degree'] = new_out_degree
    score_details['new_betweenness'] = new_betweenness
    score_details['new_ranked_pagerank'] = ranked
    score_details['new_ranked_constraints'] = constraints_ranked

    new_graph_scores[g] = score_details
    # print()
# print(len(new_graph_scores.keys())) # 6
print(new_graph_scores.keys())
print(new_graph_scores['DG_top3_betweenness_removed'].keys())
print(new_graph_scores["DG_top3_betweenness_removed"]["new_ranked_pagerank"][:3])

dict_keys(['DG_top3_in_degree_removed', 'DG_top3_out_degree_removed', 'DG_top3_betweenness_removed', 'DG_top3_pagerank_score_removed', 'DG_bottom3_constraints_removed', 'DG_top8_in_degree_removed', 'DG_top8_out_degree_removed', 'DG_top8_betweenness_removed', 'DG_top8_pagerank_score_removed', 'DG_bottom8_constraints_removed'])
dict_keys(['new_in_degree', 'new_out_degree', 'new_betweenness', 'new_ranked_pagerank', 'new_ranked_constraints'])
[('louis dreyfus company asia', 0.02176530838831755), ('standard chartered bank singapore scb', 0.014148943495698656), ('ed&f man capital markets singapore', 0.013116623244123021)]


In [15]:
# for each new graph, recompute community detection on each undirected version
new_graph_community_detection_run = {}

for g in new_graphs.keys():
    new_graph_undirected = new_graphs[g].to_undirected()
    new_graph_undirected = remove_isolated_nodes_undirected(new_graph_undirected)
    # print(new_graph_undirected.number_of_nodes(), "nodes remaining")

    seeds = [123, 234, 345, 456, 567]
    runs = {}
    mods = []

    for seed in seeds:
        run = nx.community.louvain_communities(new_graph_undirected, seed=seed)
        mod = nx.community.modularity(new_graph_undirected, run)
        runs[seed] = run
        mods.append(mod)

    best_seed = max(runs, key=lambda s: nx.community.modularity(new_graph_undirected, runs[s]))
    best_run = runs[best_seed]

    new_graph_community_detection_run[g] = {
        'all_runs': runs,
        'best_seed': best_seed,
        'best_run': best_run,
        'modularity_scores': dict(zip(seeds, mods)),
    }
print(new_graph_community_detection_run.keys())
print(new_graph_community_detection_run['DG_top3_betweenness_removed'].keys())

dict_keys(['DG_top3_in_degree_removed', 'DG_top3_out_degree_removed', 'DG_top3_betweenness_removed', 'DG_top3_pagerank_score_removed', 'DG_bottom3_constraints_removed', 'DG_top8_in_degree_removed', 'DG_top8_out_degree_removed', 'DG_top8_betweenness_removed', 'DG_top8_pagerank_score_removed', 'DG_bottom8_constraints_removed'])
dict_keys(['all_runs', 'best_seed', 'best_run', 'modularity_scores'])


In [16]:
from scipy.stats import spearmanr
from sklearn.metrics import adjusted_rand_score
import pandas as pd

DG_copy_original = DG.copy()
for node in list(DG_copy_original.nodes()):
    if DG_copy_original.in_degree(node) == 0 and DG_copy_original.out_degree(node) == 0:
        DG_copy_original.remove_node(node)

n_unique_nodes = len(set(DG_copy_original.nodes()))
node2int_original = dict(zip(set(DG_copy_original.nodes()), range(n_unique_nodes)))
int2node_original = {v: k for k, v in node2int_original.items()}

# --- save original baseline BEFORE running removal loops (protects against overwriting) ---
original_pagerank = nx.pagerank(DG_copy_original, alpha=0.85)                    # fresh, string-keyed
original_constraint_score = nx.constraint(DG_copy_original)                       # fresh, string-keyed
original_constraint_names = sorted(original_constraint_score, key=original_constraint_score.get)

original_in_degree = dict(in_degree_centrality)
original_out_degree = dict(out_degree_centrality)
original_betweenness = dict(betweenness_centrality)
original_partition = run3

n_original_connected_nodes = set(DG_copy_original.nodes())

# --- comparison helpers ---
def spearman_compare(orig_dict, new_dict, nodes):
    common = [n for n in nodes if n in orig_dict and n in new_dict]
    orig = [orig_dict[n] for n in common]
    new = [new_dict[n] for n in common]
    if len(orig) < 2:
        return None
    return spearmanr(orig, new)[0]

def count_newly_isolated(new_dict, removed_names):
    """Nodes that were connected in the ORIGINAL graph, are not among the removed top-K,
       but are missing from new_dict (i.e., became isolated as a side effect of removal)."""
    return len([n for n in n_original_connected_nodes
                if n not in removed_names and n not in new_dict])

def ari_compare(orig_partition, new_partition, nodes):
    def labels(partition):
        m = {n: i for i, comm in enumerate(partition) for n in comm}
        return [m.get(n, -1) for n in nodes]
    return adjusted_rand_score(labels(orig_partition), labels(new_partition))

def top_overlap(orig_names, new_names, top_n=10):
    return len(set(orig_names[:top_n]) & set(new_names[:top_n]))

# comparison table
def get_removed_names(name, scores_dict, topk):
    reverse = 'constraints' not in name
    ranked = sorted(scores_dict.items(), key=lambda x: x[1], reverse=reverse)[:topk]
    return [n for n, score in ranked]

rows = []
for n_k in k:
    for name, s_metric in scores.items():
        if 'constraints' in name:
            g = f"DG_bottom{n_k}_{name}_removed"
        else:
            g = f"DG_top{n_k}_{name}_removed"
        nodes = list(new_graphs[g].nodes())
        s = new_graph_scores[g]
        c = new_graph_community_detection_run[g]

        removed_names = get_removed_names(name, s_metric, n_k)  # who was actually removed for this row

        new_pagerank = dict(s['new_ranked_pagerank'])
        new_constraint_names = [n for n, score in s['new_ranked_constraints']]

        rows.append({
            'graph': g,
            'n_nodes': len(nodes),
            'spearman_in_deg': spearman_compare(original_in_degree, s['new_in_degree'], nodes),
            'spearman_out_deg': spearman_compare(original_out_degree, s['new_out_degree'], nodes),
            'spearman_betw': spearman_compare(original_betweenness, s['new_betweenness'], nodes),
            'spearman_pagerank': spearman_compare(original_pagerank, new_pagerank, nodes),
            'newly_isolated_after_removal': count_newly_isolated(new_pagerank, removed_names),
            'ARI_community': ari_compare(original_partition, c['best_run'], nodes),
            'constraint_top10_overlap': top_overlap(original_constraint_names, new_constraint_names),
        })

target_attack_summary_df = pd.DataFrame(rows)
print(target_attack_summary_df.to_string(index=False))

# save and visualize summary_df

                         graph  n_nodes  spearman_in_deg  spearman_out_deg  spearman_betw  spearman_pagerank  newly_isolated_after_removal  ARI_community  constraint_top10_overlap
     DG_top3_in_degree_removed      292         0.947540          0.968319       0.960845           0.979785                            12       0.875204                         6
    DG_top3_out_degree_removed      292         0.923843          0.969673       0.965988           0.983452                            16       0.848669                         6
   DG_top3_betweenness_removed      292         0.943957          0.969430       0.922919           0.952629                            11       0.884155                         5
DG_top3_pagerank_score_removed      295         1.000000          1.000000       1.000000           1.000000                             0       0.992547                        10
DG_bottom3_constraints_removed      295         1.000000          1.000000       1.000000           

In [17]:
target_attack_summary_df.to_csv(r"D:\commo\code\network_analysis\target_attack_summary_df.csv", index=False)

In [ ]:
# spearman_in_deg, spearman_out_deg, spearman_betw, spearman_pagerank
# among the firms that are still in the network, did their relative ranking stay the same?
# range [-1,1]; 1 = identical ranking, 0 = ranking changed and no relationship
# 0.84–1.00 : the pecking order of "who's important" barely changes even after removing the top few players

# newly_isolated_after_removal: how many firms, who used to have at least 1 connection, now have 0 connections because their only link was to one of the removed top firms
# bigger number = more collateral damage from the removal
### top 3 or top 8 firms with high centrality scores (in-degree, out-degree, betweenness) removed cause big collateral damage to the network and causes > 10 firms to have 0 recorded nodes after their removal

# ARI_community (Adjusted Rand Index)
# do firms still get grouped into the same clusters/communities as before?
# range (0,1); 1 = identical grouping, 0 = random grouping relative original
# (0.73–0.94): firms that cluster together shifted

# constraint_top10_overlap: "constraint" measures how much a firm acts as a bridge/middleman connecting otherwise-separate parts of the network (low constraint = important bridge/broker)
# of the original top-10 bridge firms, how many are still in the top-10 after removal?
# 10/10 = exact same bridge firms as before. Lower = the identity of key bridge firms changed
# top 3 or top 8 firms with high centrality scores (in-degree, out-degree, betweenness) removed changes the middleman

# conclusion:
# Removing the top 3-8 most central firms measurably affects the network, but the degree of disruption varies substantially by metric. 
# Ranking stability remains high (Spearman ρ = 0.84-1.00), showing the network's overall importance hierarchy is robust to losing its most central actors. Community structure is moderately affected (ARI = 0.73-0.94), more so at K=8 than K=3. 
# The clearest instability is in brokerage/bridge identity (constraint_top10_overlap = 4-10/10), which is highly sensitive to out-degree and betweenness-based removal specifically, but far more stable under PageRank-based removal. 
# Overall, the network shows meaningful resilience in its core structure, with the notable exception of who occupies key intermediary positions — a finding worth flagging for anyone using this analysis to prioritize outreach to "bridge" firms specifically.